# Crypto Portfolio Optimization V4 — Walk-forward

Khác V3 (V3 giữ nguyên, không sửa):
- **Walk-forward**: tại mỗi candle `t`, fit trên `n_window` candle liền trước (không gồm `t`) → weight áp cho return của candle `t`. Tải `2 × display_candles` nến: nửa đầu chỉ làm cửa sổ fit, nửa sau là phần backtest hiển thị.
- Không `bfill` (chặn look-ahead); coin chưa đủ dữ liệu trong cửa sổ bị loại khỏi cửa sổ đó.
- Chọn tài sản / tối ưu / xếp hạng đều chỉ dùng dữ liệu cửa sổ; bỏ bước chọn `max_assets` theo Sharpe in-sample.
- `mu` tính arithmetic (không ghép lãi theo giờ) và co về trung bình chéo (`mu_shrink`).
- Một `RF` duy nhất cho optimizer và metrics; có phí giao dịch trên turnover.
- Sửa đọc file trùng, bỏ ngày dư cuối index, `exp_cov span` theo timeframe.

In [ ]:
!pip install --upgrade binance-historical-data PyPortfolioOpt -q

In [ ]:
import os
import glob
import time
import requests
import zipfile
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from math import ceil
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from rich.progress import Progress
from rich.console import Console
from typing import List
from pypfopt import EfficientFrontier, CovarianceShrinkage, risk_models

# Tải dữ liệu thô từ Binance + lấy top coin (giữ nguyên từ V3)
def date_range_daily(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    current = start_dt
    dates = []
    while current <= end_dt:
        dates.append(current.strftime("%Y-%m-%d"))
        current += relativedelta(days=1)
    return dates

def date_range_monthly(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m")
    end_dt = datetime.strptime(end, "%Y-%m")
    current = start_dt
    months = []
    while current <= end_dt:
        months.append(current.strftime("%Y-%m"))
        current += relativedelta(months=1)
    return months

def download_binance_klines_fixed_symbols(
    symbols: List[str],
    intervals: List[str],
    time_period: str,          # "daily" or "monthly"
    start: str,                # "2023-01" or "2023-01-01"
    end: str,                  # "2025-11" or "2025-12-10"
    destination_dir: str = "./binance_data",
    max_workers: int = 15,
    retries: int = 3
):
    console = Console()
    base_url = "https://data.binance.vision/data"
    asset_path = "spot"

    if time_period == "daily":
        date_list = date_range_daily(start, end)
        url_template = f"{base_url}/{asset_path}/{time_period}/klines/{{symbol}}/{{interval}}/{{symbol}}-{{interval}}-{{date}}.zip"
        csv_name_template = "{symbol}-{interval}-{date}.csv"
    elif time_period == "monthly":
        date_list = date_range_monthly(start, end)
        url_template = f"{base_url}/{asset_path}/{time_period}/klines/{{symbol}}/{{interval}}/{{symbol}}-{{interval}}-{{date}}.zip"
        csv_name_template = "{symbol}-{interval}-{date}.csv"
    else:
        raise ValueError("time_period must be 'daily' or 'monthly'")

    total_files = len(symbols) * len(intervals) * len(date_list)
    success_count = 0

    def download_one(symbol: str, interval: str, date_str: str):
        nonlocal success_count
        url = url_template.format(symbol=symbol, interval=interval, date=date_str)
        csv_name = csv_name_template.format(symbol=symbol, interval=interval, date=date_str)
        local_dir = os.path.join(destination_dir, asset_path, symbol, interval)
        local_csv = os.path.join(local_dir, csv_name)
        local_zip = local_csv.replace(".csv", ".zip")

        os.makedirs(local_dir, exist_ok=True)

        # Skip if already have CSV
        if os.path.exists(local_csv):
            return True

        for attempt in range(retries + 1):
            try:
                resp = requests.get(url, timeout=20)
                if resp.status_code == 404:
                    return False
                resp.raise_for_status()

                # Save ZIP
                with open(local_zip, "wb") as f:
                    f.write(resp.content)

                # Extract first CSV
                with zipfile.ZipFile(local_zip) as zf:
                    for name in zf.namelist():
                        if name.endswith(".csv"):
                            zf.extract(name, local_dir)
                            os.rename(os.path.join(local_dir, name), local_csv)
                            break

                os.remove(local_zip)  # optional: save disk space
                return True

            except Exception as e:
                if attempt == retries:
                    console.print(f"[red]Failed {url}: {e}[/]")
                    return False
        return False

    console.print(f"[bold blue]Total files to attempt: {total_files}[/]")

    with Progress() as progress:
        task = progress.add_task("[cyan]Downloading...", total=total_files)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for symbol in symbols:
                for interval in intervals:
                    for d in date_list:
                        futures.append(executor.submit(download_one, symbol, interval, d))

            for future in as_completed(futures):
                if future.result():
                    success_count += 1
                progress.advance(task)

    console.print(f"[bold green]Done. Successfully downloaded {success_count}/{total_files} files.[/]")

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_top_coins(n_coins: int = 10) -> list:
    """
    Lấy danh sách top N đồng coin theo vốn hóa thị trường từ CoinGecko,
    loại bỏ các stablecoin, token không mong muốn và symbol trùng lặp.

    Parameters:
    - n_coins (int): Số lượng coin muốn lấy (mặc định: 1 0)

    Returns:
    - List[str]: Danh sách symbol (viết hoa), không trùng lặp, không trong blacklist
    """
    blacklist = {
        "USDT", "USDC", "DAI", "TUSD", "UST", "WBTC", "STETH", "FIGR_HELOC", "WBT", "WSTETH",
        "WBETH", "USDS", "LEO", "BSC-USD", "HYPE", "WETH", "FDUSD", "USDP", "GUSD",
        "WEETH", "XMR", "USDE", "CBBTC", "SUSDS", "CRO", "USDT0", "PYUSD", "SUSDE", "HASH",
        "MNT", "USD1", "CC", "BGB", "OKB", "M", "USDF", "BUIDL", "JITOSOL",
        "PI", "RAIN", "JLP", "HTX", "SYRUPUSDC", "USYC", "USDG", "JSOL",
        "KCS", "RLUSD", "RSETH", "BNSOL", "WBNB", "BFUSD", "RIVER", "GT", "USDD", "RETH",
        "MYX", "KAS", "LBTC", "FBTC", "USDTB", "USTB", "USDC.E", "FLR", "SYRUPUSDT", "METH",
        "WM", "JAAA", "LSETH", "KHYPE", "USDY", "OUSG", "XDC", "SOLVBTC", "EUTBL", "USDAI",
        "COCA", "CLBTC", "BDX", "WFLR", "USD0", "JTRSY", "OSETH", "A7A5", "GHO", "EZETH",
        "IP", "JUPSOL", "STABLE", "FTN", "TBTC", "CTM", "EURC", "USR", "PIPPIN", "KAU",
        "YLDS", "SIREN"
    }

    per_page = min(250, max(100, n_coins * 3))  # gọi dư hơn để bù blacklist + duplicates
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": per_page,
        "page": 1,
        "sparkline": "false"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"❌ Lỗi khi gọi API CoinGecko: {e}")
        return []

    symbols = []
    seen_symbols = set()  # theo dõi các symbol đã thêm

    for coin in data:
        symbol = coin["symbol"].upper()

        # Bỏ qua nếu đã thấy hoặc trong blacklist
        if symbol in seen_symbols or symbol in blacklist:
            continue

        symbols.append(symbol)
        seen_symbols.add(symbol)

        if len(symbols) >= n_coins:
            break

    return symbols

def get_start_date(
    timeframe: str,
    required_samples: int = 10000,
    end_date: datetime | None = None,
) -> tuple[datetime, int]:
    """
    Calculate start date to obtain at least `required_samples` candles.

    Parameters
    ----------
    timeframe : str
        e.g. '1m', '5m', '15m', '1h', '4h', '1d', '1w'
    required_samples : int
        Minimum number of candles required.
    end_date : datetime
        End date. Defaults to current UTC time.

    Returns
    -------
    start_date : datetime
    required_days : int
    """
    if end_date is None:
        end_date = datetime.utcnow()

    unit = timeframe[-1].lower()
    value = int(timeframe[:-1])

    if unit == "m":
        candles_per_day = 24 * 60 / value

    elif unit == "h":
        candles_per_day = 24 / value

    elif unit == "d":
        candles_per_day = 1 / value

    elif unit == "w":
        candles_per_day = 1 / (7 * value)

    else:
        raise ValueError(f"Unsupported timeframe: {timeframe}")

    required_days = ceil(required_samples / candles_per_day)
    start_date = end_date - timedelta(days=required_days)

    return start_date, required_days

In [ ]:
# ====== Core V4: walk-forward, không look-ahead ======
RF = 0.03  # một risk-free duy nhất cho optimizer + metrics
EUR_ASSETS = {"EURUSDT", "EURIUSDT"}


def tf_delta(tf: str) -> pd.Timedelta:
    return pd.Timedelta(tf.replace('m', 'min'))


def periods_per_year(tf: str) -> float:
    return 365 * 24 * 3600 / tf_delta(tf).total_seconds()


def load_prices(tickers, start, end, timeframe, data_dir="binance_data"):
    """Close price của mọi ticker trên 1 index chung. Chỉ ffill (causal), KHÔNG bfill:
    coin niêm yết muộn giữ NaN ở đoạn trước khi có giá."""
    delta = tf_delta(timeframe)
    idx = pd.date_range(pd.Timestamp(start, tz="UTC"),
                        pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1) - delta, freq=delta)
    cols = {}
    for t in tickers:
        files = glob.glob(os.path.join(data_dir, "spot", t, timeframe, "*.csv"))  # mỗi file đọc 1 lần
        if not files:
            print(f"⚠️  Bỏ {t}: không có file")
            continue
        df = pd.concat([read_csv_file(f) for f in files])[["open_time", "Close"]]
        df["open_time"] = pd.to_datetime(df["open_time"], utc=True)
        s = df.drop_duplicates("open_time").set_index("open_time")["Close"]
        cols[t] = s[s > 0].reindex(idx)
    return pd.DataFrame(cols).ffill()


def cov_matrix(win, P, model, span):
    shrink = lambda: CovarianceShrinkage(win, frequency=P).ledoit_wolf()
    if model == "shrinkage":
        return shrink()
    if model == "sample":
        return risk_models.sample_cov(win, frequency=P)
    S = risk_models.exp_cov(win, frequency=P, span=span)
    return S if model == "exp" else 0.5 * S + 0.5 * shrink()


def max_drawdown(r: pd.Series) -> float:
    cum = (1 + r).cumprod()
    return (cum / cum.cummax() - 1).min()


def optimize_window(win, P, cfg, max_assets):
    """Chọn tài sản + tối ưu CHỈ trên `win` (dữ liệu quá khứ). Trả {strategy: weights}."""
    rets = win.pct_change().dropna()
    mu = rets.mean() * P                      # arithmetic, không ghép lãi theo giờ
    sd = rets.std() * np.sqrt(P)
    ranked = (mu / sd).sort_values(ascending=False).index

    req = {a: m for a, m in cfg["required_assets"].items() if a in win.columns}
    sel = list(dict.fromkeys(list(req) + list(ranked)))[:max_assets]
    lo = [req.get(a, cfg["min_weight"]) for a in sel]
    hi = [cfg["eur_max_weight"] if a in EUR_ASSETS else 1.0 for a in sel]
    if sum(lo) > 1 or any(l > h for l, h in zip(lo, hi)):
        return {}

    s = cfg["mu_shrink"]                      # co mu về trung bình chéo -> bớt nhiễu
    mu_s = (1 - s) * mu[sel] + s * mu[sel].mean()
    S = cov_matrix(win[sel], P, cfg["risk_model"], cfg["span"])
    bounds = list(zip(lo, hi))

    def solve(kind):
        ef = EfficientFrontier(mu_s, S, weight_bounds=bounds)
        try:
            if kind == "sharpe":
                ef.max_sharpe(risk_free_rate=RF)
            elif kind == "optimal":
                ef.efficient_return(cfg["target_return"])
            else:
                ef.min_volatility()
        except Exception:                     # vd: mọi mu < RF, target vượt khả thi
            ef = EfficientFrontier(mu_s, S, weight_bounds=bounds)
            ef.min_volatility()
        return dict(ef.clean_weights())

    return {k: solve(k) for k in ("sharpe", "volatility", "optimal")}


def make_targets(win, P, cfg, max_assets):
    rets = win.pct_change().dropna()
    cols = list(rets.columns)
    out = {"equal_weight": {c: 1 / len(cols) for c in cols}}

    inv = 1 / rets.std().replace(0, np.nan).dropna()
    out["risk_parity"] = (inv / inv.sum()).to_dict()

    # equal absolute drawdown: phần dư (cash_asset) không tham gia ngân sách rủi ro
    cash = cfg["cash_asset"]
    mdds = {c: abs(max_drawdown(rets[c])) for c in cols if c != cash}
    mdds = {c: m for c, m in mdds.items() if m > 0}
    for tgt in map(abs, cfg["mdd_targets"]):
        if not mdds:
            break
        w = {c: (tgt / len(mdds)) / m for c, m in mdds.items()}
        tot = sum(w.values())
        if tot > 1:
            w = {c: x / tot for c, x in w.items()}
            tot = 1.0
        if cash in cols:
            w[cash] = 1 - tot                 # nếu thiếu cash_asset: phần dư để tiền mặt (0%)
        out[f"eq_drawdown_{int(tgt * 100)}"] = w

    try:
        out.update({f"opt_{k}": v for k, v in optimize_window(win, P, cfg, max_assets).items()})
    except Exception as e:
        print(f"  ⚠️ optimize lỗi, giữ weight cũ: {e}")
    return out


def walk_forward(prices, n_window, timeframe, cfg, max_assets, rebalance_every=1, fee=0.001):
    """Tại candle t: fit trên prices[t-n_window : t] (không gồm t) -> áp weight cho return của candle t.
    Giữa 2 lần rebalance weight trôi theo giá. Phí = fee * turnover."""
    P = periods_per_year(timeframe)
    cols = list(prices.columns)
    pos = {c: i for i, c in enumerate(cols)}
    rets = prices.pct_change()
    W, R, hist = {}, {}, {}
    total = len(prices) - n_window
    t0 = time.time()

    for k, t in enumerate(range(n_window, len(prices))):
        turn = {}
        if k % rebalance_every == 0:
            targets = make_targets(prices.iloc[t - n_window:t].dropna(axis=1), P, cfg, max_assets)
            if k == 0 and "BTCUSDT" in pos:
                targets["btc_hold"] = {"BTCUSDT": 1.0}   # buy & hold, không rebalance
            for name, tw in targets.items():
                vec = np.zeros(len(cols))
                for a, x in tw.items():
                    vec[pos[a]] = x
                turn[name] = np.abs(vec - W.get(name, np.zeros(len(cols)))).sum()
                W[name] = vec
                hist.setdefault(name, {})[prices.index[t]] = pd.Series(tw)
        r = rets.iloc[t].fillna(0).to_numpy()
        for name, w in W.items():
            pr = float(w @ r)
            R.setdefault(name, []).append(pr - fee * turn.get(name, 0.0))
            W[name] = w * (1 + r) / (1 + pr)
        if (k + 1) % 500 == 0 or k + 1 == total:
            print(f"  {k + 1}/{total} candles - {time.time() - t0:.0f}s")

    idx = prices.index[n_window:]
    returns = pd.DataFrame({n: pd.Series(v, index=idx[-len(v):]) for n, v in R.items()})
    return returns, {n: pd.DataFrame(h).T.fillna(0) for n, h in hist.items()}


def summarize(returns, timeframe, weights_hist):
    P = periods_per_year(timeframe)
    rows = {}
    for name, r in returns.items():
        eq = (1 + r).cumprod()
        sd = r.std()
        wh = weights_hist.get(name)
        # turnover/năm = tổng |Δweight| giữa các lần rebalance (không tính trôi giá)
        tn = wh.diff().abs().sum().sum() if wh is not None and len(wh) > 1 else 0.0
        rows[name] = {
            "return_cagr": eq.iloc[-1] ** (P / len(r)) - 1,
            "volatility": sd * np.sqrt(P),
            "sharpe": (r.mean() - RF / P) / sd * np.sqrt(P) if sd > 0 else 0.0,
            "max_drawdown": (eq / eq.cummax() - 1).min(),
            "turnover_per_year": tn * P / len(r),
        }
    return pd.DataFrame(rows).T.sort_values("sharpe", ascending=False)


In [ ]:
timeframe = '1h'
display_candles = 10000          # số nến hiển thị/backtest; cửa sổ fit cũng = chừng này nến trước đó
n_window = display_candles
rebalance_every = 1              # 1 = tính lại weight mỗi candle (chậm); tăng lên (vd 24) để nhanh hơn
fee = 0.001                      # phí/turnover mỗi lần rebalance
max_assets = 6

cfg = dict(
    required_assets={'EURUSDT': 0.15, 'PAXGUSDT': 0.09, 'BTCUSDT': 0.06, 'ETHUSDT': 0.04},
    min_weight=0.01,
    eur_max_weight=0.34,
    target_return=0.05,
    risk_model="exp_shrink",     # shrinkage | sample | exp | exp_shrink
    mu_shrink=0.5,               # 0 = mu thô, 1 = mọi mã cùng mu
    cov_span_days=60,
    mdd_targets=(-0.08, -0.13, -0.21, -0.34, -0.55),
    cash_asset="EURUSDT",
)
cfg["span"] = int(cfg["cov_span_days"] * 24 * 3600 / tf_delta(timeframe).total_seconds())

list_tickers = get_top_coins(8)
tickers_binance = ["EURUSDT", "PAXGUSDT"] + [s + "USDT" for s in list_tickers]
print("Top tickers:", tickers_binance)

end_date = date.today() - timedelta(days=1)
start_date, days = get_start_date(timeframe, display_candles + n_window + 24, end_date)
start_str, end_str = start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")
print(f"{timeframe} -> {days} days: {start_str} .. {end_str}")

download_binance_klines_fixed_symbols(
    symbols=tickers_binance, intervals=[timeframe], time_period="daily",
    start=start_str, end=end_str, max_workers=15, retries=3,
)

In [ ]:
price_df = load_prices(tickers_binance, start_str, end_str, timeframe)
price_df = price_df.iloc[-(display_candles + n_window):]       # đúng 2N nến cuối
n_window = min(n_window, len(price_df) - 1)
print(f"{len(price_df)} candles, cửa sổ fit = {n_window}, backtest = {len(price_df) - n_window}")
print("Số giá hợp lệ / mã:"); print(price_df.notna().sum().to_string())

In [ ]:
returns_wf, weights_hist = walk_forward(
    price_df, n_window, timeframe, cfg, max_assets,
    rebalance_every=rebalance_every, fee=fee,
)
summary = summarize(returns_wf, timeframe, weights_hist)
(summary * [100, 100, 1, 100, 1]).round(2)

In [ ]:
fig = go.Figure()
for name, r in returns_wf.items():
    fig.add_trace(go.Scatter(x=r.index, y=(1 + r).cumprod(), mode="lines", name=name,
                             line=dict(width=3 if name.startswith("opt_") or name == "btc_hold" else 1)))
fig.update_layout(title="Walk-forward equity (net of fees)", hovermode="x unified")
fig.show()

In [ ]:
# Weight theo thời gian của một chiến lược tối ưu
name = "opt_sharpe"
wh = weights_hist[name]
fig = go.Figure([go.Scatter(x=wh.index, y=wh[c], stackgroup="w", name=c) for c in wh.columns])
fig.update_layout(title=f"Weights - {name}", hovermode="x unified")
fig.show()